# EyelinerCNN — Google Colab 訓練ノートブック

**前提条件:**
- GitHubにコードがプッシュ済み
- Google Drive の `My Drive/EyelinerDetecter/data/images/` に画像がアップロード済み
- ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択済み

---

## 0. 設定（最初に変更してください）

In [ ]:
# ===== ここだけ変更 =====
GITHUB_USER  = "kurumikurukuri"          # GitHubのユーザー名
REPO_NAME    = "EyelinerDetecter"         # リポジトリ名
BRANCH       = "feat/re_re_rearchitecture" # ブランチ名
GITHUB_TOKEN = ""                         # GitHubのPersonal Access Token（プライベートリポジトリの場合）
DRIVE_IMAGE_DIR = "EyelinerDetecter/data/images"  # Drive上の画像フォルダパス
# =======================

## 1. GPU 確認

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPUが割り当てられていません")
    print("ランタイム → ランタイムのタイプを変更 → T4 GPU を選択してください")

## 2. リポジトリのクローン

In [ ]:
import os

REPO_DIR = f"/content/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    # 再実行時は最新コードに更新
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
print(f"作業ディレクトリ: {os.getcwd()}")

## 3. Google Drive マウント & 画像リンク

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
src = f"{DRIVE_ROOT}/{DRIVE_IMAGE_DIR}"
dst = f"{REPO_DIR}/data/images"

os.makedirs(dst, exist_ok=True)

# シンボリックリンクで参照（コピー不要・高速）
!ln -sf {src}/* {dst}/

n_images = len(list(os.scandir(dst)))
print(f"画像リンク数: {n_images}")
if n_images == 0:
    print("WARNING: 画像が見つかりません。Drive上のパスを確認してください")
    print(f"  期待パス: {src}")

## 4. 依存パッケージのインストール

In [ ]:
# mediapipe のインストールに2〜3分かかります
!pip install -q -r requirements.txt

# 動作確認
import importlib
for pkg in ["torch", "torchvision", "mediapipe", "cv2", "sklearn"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK: {pkg}")
    except ImportError:
        print(f"  NG: {pkg}")

## 5. データ疎通確認

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, ".")
from annotator import db as _db

samples = _db.get_annotated_samples(Path("data/samples.db"))
manifest = pd.DataFrame(samples)
print(f"annotated samples (DB): {len(manifest)} 件")

# 画像ファイルが実際に存在するか確認
missing = []
for img_path in manifest["image_path"].head(20):
    if not os.path.exists(img_path):
        missing.append(img_path)

if missing:
    print(f"WARNING: 画像が見つからないサンプルあり: {missing[:3]} ...")
else:
    print("先頭20件の画像パス: OK")

## 6. EXP012 manifest 準備（初回のみ）

EXP011 と同一データ分割を使用します。

In [ ]:
import os

# EXP011 の manifest をコピーして流用（同一データ分割）
!cp experiments/EXP011_hparam_sweep/manifests/manifest_train.csv \n   experiments/EXP012_patch_grid_sweep/manifests/manifest_train.csv
!cp experiments/EXP011_hparam_sweep/manifests/manifest_test.csv \n   experiments/EXP012_patch_grid_sweep/manifests/manifest_test.csv

# 確認
import pandas as pd
train = pd.read_csv('experiments/EXP012_patch_grid_sweep/manifests/manifest_train.csv')
test  = pd.read_csv('experiments/EXP012_patch_grid_sweep/manifests/manifest_test.csv')
print(f'manifest_train: {len(train)} 件')
print(f'manifest_test : {len(test)} 件')

---
## 7. 実験実行（グループ別 → 都度保存）

グループごとに「実行 → Drive保存」を繰り返します。  
1グループ終わるたびに保存セルを実行してください。

### 7-A. グループA 実行

In [ ]:
!python experiments/EXP012_patch_grid_sweep/scr/train_exp012.py --group A

### 7-A. グループA → Drive 保存

In [ ]:
import shutil, os
from pathlib import Path
from datetime import datetime

EXP_NUM = "012"
GROUP   = "A"

exp_dirs = list(Path("experiments").glob(f"EXP{EXP_NUM}_*"))
if not exp_dirs:
    print(f"EXP{EXP_NUM} が見つかりません")
else:
    exp_dir = exp_dirs[0]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    save_dst = f"{DRIVE_ROOT}/eyelinerCNN/outputs/EXP{EXP_NUM}_{GROUP}_{timestamp}"
    group_outputs = list((exp_dir / "outputs").glob(f"exp{EXP_NUM}_{GROUP}_*"))
    if not group_outputs:
        print(f"グループ {GROUP} の出力が見つかりません")
    else:
        os.makedirs(save_dst, exist_ok=True)
        for out_dir in group_outputs:
            shutil.copytree(str(out_dir), f"{save_dst}/{out_dir.name}", dirs_exist_ok=True)
        print(f"グループ {GROUP} 保存完了: {save_dst} ({len(group_outputs)} 件)")

### 7-B. グループB 実行

In [ ]:
!python experiments/EXP012_patch_grid_sweep/scr/train_exp012.py --group B

### 7-B. グループB → Drive 保存

In [ ]:
EXP_NUM = "012"
GROUP   = "B"

exp_dirs = list(Path("experiments").glob(f"EXP{EXP_NUM}_*"))
if not exp_dirs:
    print(f"EXP{EXP_NUM} が見つかりません")
else:
    exp_dir = exp_dirs[0]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    save_dst = f"{DRIVE_ROOT}/eyelinerCNN/outputs/EXP{EXP_NUM}_{GROUP}_{timestamp}"
    group_outputs = list((exp_dir / "outputs").glob(f"exp{EXP_NUM}_{GROUP}_*"))
    if not group_outputs:
        print(f"グループ {GROUP} の出力が見つかりません")
    else:
        os.makedirs(save_dst, exist_ok=True)
        for out_dir in group_outputs:
            shutil.copytree(str(out_dir), f"{save_dst}/{out_dir.name}", dirs_exist_ok=True)
        print(f"グループ {GROUP} 保存完了: {save_dst} ({len(group_outputs)} 件)")

### 7-C. グループC 実行

In [ ]:
!python experiments/EXP012_patch_grid_sweep/scr/train_exp012.py --group C

### 7-C. グループC → Drive 保存

In [ ]:
EXP_NUM = "012"
GROUP   = "C"

exp_dirs = list(Path("experiments").glob(f"EXP{EXP_NUM}_*"))
if not exp_dirs:
    print(f"EXP{EXP_NUM} が見つかりません")
else:
    exp_dir = exp_dirs[0]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    save_dst = f"{DRIVE_ROOT}/eyelinerCNN/outputs/EXP{EXP_NUM}_{GROUP}_{timestamp}"
    group_outputs = list((exp_dir / "outputs").glob(f"exp{EXP_NUM}_{GROUP}_*"))
    if not group_outputs:
        print(f"グループ {GROUP} の出力が見つかりません")
    else:
        os.makedirs(save_dst, exist_ok=True)
        for out_dir in group_outputs:
            shutil.copytree(str(out_dir), f"{save_dst}/{out_dir.name}", dirs_exist_ok=True)
        print(f"グループ {GROUP} 保存完了: {save_dst} ({len(group_outputs)} 件)")

### 7-D. グループD 実行

In [ ]:
!python experiments/EXP012_patch_grid_sweep/scr/train_exp012.py --group D

### 7-D. グループD → Drive 保存

In [ ]:
EXP_NUM = "012"
GROUP   = "D"

exp_dirs = list(Path("experiments").glob(f"EXP{EXP_NUM}_*"))
if not exp_dirs:
    print(f"EXP{EXP_NUM} が見つかりません")
else:
    exp_dir = exp_dirs[0]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    save_dst = f"{DRIVE_ROOT}/eyelinerCNN/outputs/EXP{EXP_NUM}_{GROUP}_{timestamp}"
    group_outputs = list((exp_dir / "outputs").glob(f"exp{EXP_NUM}_{GROUP}_*"))
    if not group_outputs:
        print(f"グループ {GROUP} の出力が見つかりません")
    else:
        os.makedirs(save_dst, exist_ok=True)
        for out_dir in group_outputs:
            shutil.copytree(str(out_dir), f"{save_dst}/{out_dir.name}", dirs_exist_ok=True)
        print(f"グループ {GROUP} 保存完了: {save_dst} ({len(group_outputs)} 件)")

## 8. モデルファイルをローカルにダウンロード（オプション）

In [ ]:
from google.colab import files
import glob

EXP_NUM = "012"

# 全 .pt ファイルをダウンロード
pt_files = glob.glob(f"experiments/EXP{EXP_NUM}_*/outputs/**/*.pt", recursive=True)
print(f"{len(pt_files)} 件のモデルファイルが見つかりました")

for pt in pt_files:
    print(f"  ダウンロード: {pt}")
    files.download(pt)